# Preparación de los datos y vistas minables

En este notebook aplicamos lo que decidimos en `01_exploracion_calidad_ml.ipynb` (tabla 3.6) y armamos
las tablas que van a usar los modelos.

Está dividido en dos partes:
- **Parte A, limpieza:** generamos versiones limpias de `users`, `acoustic_features` y `events` y las
  guardamos en `data/clean/`.
- **Parte B, vistas minables:** con los datos limpios armamos una tabla para cada técnica y las guardamos
  en `data/vistas/`.

| Técnica | Vista | Cada fila es... |
|---|---|---|
| Clustering de usuarios | `vista_usuarios` | un usuario |
| Sistema de recomendación | `vista_usuario_artista` | un usuario con un artista |
| Reglas de asociación | `vista_sesiones` | un artista dentro de una sesión de escucha |

Usamos una tabla distinta para cada técnica porque cada una necesita analizar algo diferente
(usuarios, pares usuario-artista o sesiones).

## 1. Preparación del entorno

In [1]:
# --- Librerías ---
import os
import duckdb
import pandas as pd

# --- Conexión a DuckDB ---
con = duckdb.connect()
con.execute("SET memory_limit = '4GB'")
con.execute("SET TimeZone = 'UTC'")

# Carpeta donde DuckDB guarda datos temporales cuando una consulta no entra en memoria
con.execute("SET temp_directory = '../.tmp'")

# --- Carpetas de salida ---
# Las rutas empiezan con ../ porque el notebook está en la carpeta mineria/
# y los datos están en la carpeta principal del repo.
os.makedirs("../data/clean", exist_ok=True)
os.makedirs("../data/vistas", exist_ok=True)

# --- Vistas sobre los archivos originales (igual que en el notebook 01) ---
ruta = "../data/parquet/"
con.execute(f"CREATE OR REPLACE VIEW events            AS SELECT * FROM '{ruta}events.parquet'")
con.execute(f"CREATE OR REPLACE VIEW users             AS SELECT * FROM '{ruta}users.parquet'")
con.execute(f"CREATE OR REPLACE VIEW acoustic_features AS SELECT * FROM '{ruta}acoustic_features.parquet'")

# --- Función auxiliar: ejecuta una consulta y devuelve un DataFrame ---
def q(consulta):
    return con.sql(consulta).df()

q("SELECT 'todo listo' AS estado")

,estado
0,todo listo


---
# Parte A: limpieza

Aplicamos las decisiones de la tabla 3.6 del notebook 01. No borramos usuarios ni canciones: los
valores imposibles los pasamos a nulo, y solo eliminamos las escuchas que no pueden ser reales
(la de 1970 y las de los días con más de 720 escuchas).

Cada tabla limpia la guardamos como parquet con `COPY ... TO`, así no hay que repetir la limpieza
cada vez que abrimos un notebook.

### A.1 Usuarios
- Las edades fuera de 10–90 (incluido el -1) pasan a nulo.
- El género "n" o vacío pasa a "no informado".
- El país "UK" pasa a "GB", que es el código estándar del Reino Unido.

In [2]:
# CASE funciona como un if: si se cumple la condición deja el valor, si no, pone otro.
con.execute("""
    COPY (
        SELECT user_id,
               CASE WHEN country = 'UK' THEN 'GB' ELSE country END        AS country,
               CASE WHEN age BETWEEN 10 AND 90 THEN age ELSE NULL END       AS age,
               CASE WHEN gender IN ('m', 'f') THEN gender
                    ELSE 'no informado' END                                AS gender,
               playcount,
               registered_unixtime
        FROM users
    ) TO '../data/clean/users_clean.parquet' (FORMAT parquet)
""")

# Control: cuántas edades quedaron nulas y cómo quedó el género
q("""
    SELECT COUNT(*)                                    AS usuarios,
           COUNT(*) FILTER (WHERE age IS NULL)          AS edad_nula,
           COUNT(*) FILTER (WHERE gender = 'no informado') AS genero_no_informado,
           COUNT(*) FILTER (WHERE country = 'UK')       AS quedan_uk
    FROM '../data/clean/users_clean.parquet'
""")

,usuarios,edad_nula,genero_no_informado,quedan_uk
0,55190,12481,4747,0


Quedaron los 55.190 usuarios, 12.481 con la edad en nulo, 4.747 con género "no informado" y ninguno
con "UK", que son los números que esperábamos según el notebook 01.

### A.2 Características de audio
- `tempo` = 0 pasa a nulo (una canción no puede tener 0 BPM).
- `loudness` = -60 dB pasa a nulo (es el mínimo de Spotify, sería silencio total).
- Sacamos `key` y `mode`, porque son categóricas (la nota y si es mayor o menor) y no las usamos
  en ninguna técnica.

In [3]:
# NULLIF(columna, valor) devuelve nulo cuando la columna es igual a ese valor
con.execute("""
    COPY (
        SELECT track_id,
               danceability,
               energy,
               NULLIF(loudness, -60) AS loudness,
               speechiness,
               acousticness,
               instrumentalness,
               liveness,
               valence,
               NULLIF(tempo, 0)      AS tempo
        FROM acoustic_features
    ) TO '../data/clean/acoustic_clean.parquet' (FORMAT parquet)
""")

# Control: cantidad de nulos en las dos columnas que cambiamos
q("""
    SELECT COUNT(*)                  AS canciones,
           COUNT(*) - COUNT(tempo)    AS tempo_nulo,
           COUNT(*) - COUNT(loudness) AS loudness_nulo
    FROM '../data/clean/acoustic_clean.parquet'
""")

,canciones,tempo_nulo,loudness_nulo
0,3471884,2651,209


Quedaron 2.651 canciones con tempo nulo y 209 con loudness nulo, igual que en el notebook 01.

### A.3 Escuchas
Sacamos:
1. Las escuchas anteriores al 1/1/2005 (es solo la de 1970). `1104537600` es esa fecha en unix time.
2. Todas las escuchas de los días en que un usuario pasó las 720 escuchas, que según la regla de
   Last.fm no se pueden hacer en 24 horas.

Para agrupar por día usamos `"timestamp" // 86400`. Un día tiene 86.400 segundos, así que la división
entera da el número de día (en UTC). Da lo mismo que `date_trunc('day', ...)`, pero es más rápido
porque no tiene que convertir 351 millones de números a fechas.

Para sacar las escuchas de los días anómalos hacemos un `LEFT JOIN` y nos quedamos con las que
**no** encontraron pareja (`a.user_id IS NULL`).

Esta celda tarda varios minutos porque recorre toda la tabla de escuchas.

In [4]:
con.execute("""
    COPY (
        -- 1) Escuchas de cada usuario en cada día
        WITH dias AS (
            SELECT user_id, "timestamp" // 86400 AS dia, COUNT(*) AS escuchas_dia
            FROM events
            GROUP BY user_id, dia
        ),
        -- 2) Los días con más de 720 escuchas
        dias_anomalos AS (
            SELECT user_id, dia
            FROM dias
            WHERE escuchas_dia > 720
        )
        -- 3) Todas las escuchas menos las de esos días y la de 1970
        SELECT e.user_id, e.artist_id, e.album_id, e.track_id, e."timestamp"
        FROM events e
        LEFT JOIN dias_anomalos a
               ON e.user_id = a.user_id
              AND e."timestamp" // 86400 = a.dia
        WHERE a.user_id IS NULL
          AND e."timestamp" >= 1104537600
    ) TO '../data/clean/events_clean.parquet' (FORMAT parquet, COMPRESSION zstd)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

### A.4 Control de la limpieza

In [5]:
antes   = con.execute("SELECT COUNT(*) FROM events").fetchone()[0]
despues = con.execute("SELECT COUNT(*) FROM '../data/clean/events_clean.parquet'").fetchone()[0]

print(f"Escuchas originales: {antes:>15,}")
print(f"Escuchas limpias:    {despues:>15,}")
print(f"Eliminadas:          {antes - despues:>15,}  ({(antes - despues) / antes * 100:.2f}%)")

Escuchas originales:     351,469,333
Escuchas limpias:        350,741,953
Eliminadas:                  727,380  (0.21%)


Se eliminaron 727.380 escuchas (0,21%): 727.379 de los días anómalos más la de 1970. Coincide con lo
que habíamos calculado en el notebook 01.

**Resumen de la parte A:** quedaron tres archivos en `data/clean/`: `users_clean.parquet`,
`acoustic_clean.parquet` y `events_clean.parquet`. De acá en adelante usamos solo estos.

---
# Parte B: vistas minables

### B.0 Datos de partida
Creamos vistas sobre los archivos limpios y una tabla auxiliar con la cantidad de **oyentes distintos
de cada artista**. La usamos en las tres vistas:
- en `vista_usuarios`, para medir qué tan conocido es lo que escucha cada usuario;
- en las otras dos, para quedarnos con los artistas más escuchados.

La guardamos como tabla temporal (`TEMP TABLE`) porque contar los oyentes de cada artista sobre toda
la tabla de escuchas tarda, y así lo hacemos una sola vez.

In [6]:
con.execute("CREATE OR REPLACE VIEW events_clean   AS SELECT * FROM '../data/clean/events_clean.parquet'")
con.execute("CREATE OR REPLACE VIEW acoustic_clean AS SELECT * FROM '../data/clean/acoustic_clean.parquet'")
con.execute("CREATE OR REPLACE VIEW users_clean    AS SELECT * FROM '../data/clean/users_clean.parquet'")

# Oyentes distintos por artista (una fila por artista)
con.execute("""
    CREATE OR REPLACE TEMP TABLE oyentes_artista AS
    SELECT artist_id, COUNT(DISTINCT user_id) AS oyentes
    FROM events_clean
    GROUP BY artist_id
""")

q("SELECT * FROM oyentes_artista ORDER BY oyentes DESC LIMIT 10")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,artist_id,oyentes
0,320,19886
1,1602,19140
2,46,17926
3,3,17115
4,4115,17009
5,295,16695
6,27,16338
7,185,16191
8,924,15989
9,470,15548


### B.1 Vista para clustering: `vista_usuarios`

**Cada fila es un usuario**, descrito por *cómo* escucha música.

**Filtro:** usuarios con al menos 100 escuchas de canciones con audio. Con menos, los promedios
dependen de muy pocas canciones (ver 4.2.a del notebook 01).

**Variables que entran al modelo:**

| Variable | Cómo se calcula | Qué mide |
|---|---|---|
| `energy_media`, `valence_media`, `dance_media`, `acoustic_media`, `instrumental_media`, `tempo_medio` | promedio por usuario | cómo suena en general lo que escucha |
| `energy_std`, `valence_std` | desvío estándar por usuario | si escucha siempre lo mismo o cosas variadas |
| `artistas_distintos` | cantidad de artistas distintos | qué tan amplios son sus gustos |
| `pct_artista_top` | escuchas a su artista favorito / total | si se concentra en un artista o reparte |
| `popularidad_media` | promedio del log10 de los oyentes de cada artista | si escucha artistas conocidos o de nicho |

**Datos del usuario que no entran al modelo:** `total_escuchas`, `country`, `age` y `gender`. Los usamos
después para describir los grupos.

Dejamos afuera `loudness` porque tiene una correlación de 0,78 con `energy` (ver 4.3.b del notebook 01),
y `speechiness` y `liveness` porque casi no varían.

Para la popularidad usamos el log10 de los oyentes porque va de 1 a casi 20 mil. Sin el logaritmo,
los pocos artistas enormes pesarían demasiado en el promedio.

La consulta está armada en partes con `WITH`: cada parte es una tabla intermedia con nombre.

In [7]:
consulta_usuarios = """
-- 1) Cada escucha con el audio de la canción y la popularidad del artista.
--    El JOIN con acoustic_clean deja solo las canciones que tienen audio.
WITH escuchas AS (
    SELECT e.user_id, e.artist_id,
           a.energy, a.valence, a.danceability, a.acousticness,
           a.instrumentalness, a.tempo,
           LOG10(o.oyentes) AS popularidad
    FROM events_clean e
    JOIN acoustic_clean  a ON e.track_id  = a.track_id
    JOIN oyentes_artista o ON e.artist_id = o.artist_id
),

-- 2) Escuchas de cada usuario a cada artista; el máximo es su artista favorito
por_artista AS (
    SELECT user_id, artist_id, COUNT(*) AS n
    FROM escuchas
    GROUP BY user_id, artist_id
),
favorito AS (
    SELECT user_id, MAX(n) AS escuchas_top
    FROM por_artista
    GROUP BY user_id
),

-- 3) Resumen por usuario, solo los que tienen 100 escuchas o más
resumen AS (
    SELECT user_id,
           COUNT(*)                  AS total_escuchas,
           COUNT(DISTINCT artist_id) AS artistas_distintos,
           AVG(energy)               AS energy_media,
           STDDEV(energy)            AS energy_std,
           AVG(valence)              AS valence_media,
           STDDEV(valence)           AS valence_std,
           AVG(danceability)         AS dance_media,
           AVG(acousticness)         AS acoustic_media,
           AVG(instrumentalness)     AS instrumental_media,
           AVG(tempo)                AS tempo_medio,
           AVG(popularidad)          AS popularidad_media
    FROM escuchas
    GROUP BY user_id
    HAVING COUNT(*) >= 100
)

-- 4) Se agrega el % del favorito y los datos del usuario.
--    LEFT JOIN para no perder a nadie aunque le falte algún dato personal.
SELECT r.*,
       f.escuchas_top * 1.0 / r.total_escuchas AS pct_artista_top,
       u.country, u.age, u.gender
FROM resumen r
JOIN favorito f         ON r.user_id = f.user_id
LEFT JOIN users_clean u ON r.user_id = u.user_id
"""

con.execute(f"COPY ({consulta_usuarios}) TO '../data/vistas/vista_usuarios.parquet' (FORMAT parquet)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

**Control de la vista:** cuántos usuarios quedaron, si hay nulos y qué valores toma cada variable.
`total_escuchas` cuenta solo las escuchas de canciones con audio.

In [8]:
vista_usuarios = q("SELECT * FROM '../data/vistas/vista_usuarios.parquet'")

print(f"Usuarios en la vista: {len(vista_usuarios):,} de 55.190")
print("\nNulos por columna:")
print(vista_usuarios.isna().sum())

vista_usuarios.describe().T.round(3)

Usuarios en la vista: 53,289 de 55.190

Nulos por columna:
user_id                   0
total_escuchas            0
artistas_distintos        0
energy_media              0
energy_std                0
valence_media             0
valence_std               0
dance_media               0
acoustic_media            0
instrumental_media        0
tempo_medio               0
popularidad_media         0
pct_artista_top           0
country                   0
age                   11961
gender                    0
dtype: int64


,count,mean,std,min,25%,50%,75%,max
user_id,53289.0,23144265.899623,16094451.221663,384.0,8065274.0,19857764.0,38124557.0,50900118.0
total_escuchas,53289.0,6580.227345,9081.736118,100.0,1534.0,5310.0,8865.0,345014.0
artistas_distintos,53289.0,337.370058,366.793969,1.0,113.0,229.0,434.0,13354.0
energy_media,53289.0,0.692687,0.117028,0.102642,0.615357,0.694493,0.778673,0.964863
energy_std,53289.0,0.213505,0.039268,0.010329,0.188542,0.218539,0.241184,0.406878
valence_media,53289.0,0.424335,0.097475,0.043585,0.361969,0.429971,0.489383,0.844944
valence_std,53289.0,0.230226,0.027479,0.045913,0.215965,0.23517,0.24894,0.364258
dance_media,53289.0,0.495884,0.0791,0.103918,0.444998,0.499692,0.548037,0.861681
acoustic_media,53289.0,0.211274,0.132327,0.000147,0.108324,0.192218,0.288744,0.980609
instrumental_media,53289.0,0.214657,0.153411,0.0,0.093846,0.180581,0.30571,0.935433


**B.1 Observaciones:**
- Quedaron 53.289 usuarios de 55.190 (96,6%). Con el mínimo de 100 escuchas perdemos muy pocos.
- La única columna con nulos es `age` (11.961), que no entra al modelo.
- Las escalas son muy distintas: `tempo_medio` ronda 124, `artistas_distintos` va de 1 a 13.354 y el resto
  de las variables va de 0 a 1. Antes de K-means hay que estandarizar.
- `artistas_distintos` está muy estirada hacia la derecha: la mediana es 229 pero el máximo es 13.354.
  En el clustering conviene usar su logaritmo.
- `tempo_medio` varía muy poco entre usuarios (el desvío es 5 BPM): al promediar muchas canciones,
  todos terminan cerca de 124. Puede que esta variable ayude poco a separar grupos.
- Hay usuarios con `pct_artista_top` = 1, o sea que escuchan a un solo artista. Son casos raros, pero
  válidos.

En el notebook de clustering hacemos los histogramas y la correlación de estas variables.

### B.2 Vista para el sistema de recomendación: `vista_usuario_artista`

**Cada fila es un usuario con un artista**, y la cantidad de veces que lo escuchó.

**Filtros:**
- **Los mismos usuarios que en `vista_usuarios`**, para trabajar con la misma gente en las dos técnicas
  (y poder usar los clusters para comparar el recomendador).
- **Los 10.000 artistas con más oyentes.** Con todos los artistas (unos 338 mil) la matriz
  usuario × artista quedaría casi vacía, y a un artista que escucharon dos personas no se lo puede
  recomendar bien.

Acá solo armamos la tabla. La matriz y la separación en entrenamiento y prueba las hacemos en el
notebook del recomendador.

In [9]:
consulta_ua = """
-- Los 10.000 artistas con más oyentes distintos
WITH top_artistas AS (
    SELECT artist_id
    FROM oyentes_artista
    ORDER BY oyentes DESC
    LIMIT 10000
),
-- Los usuarios que quedaron en la vista de clustering
usuarios_ok AS (
    SELECT user_id FROM '../data/vistas/vista_usuarios.parquet'
)
-- Cuántas veces escuchó cada usuario a cada uno de esos artistas
SELECT e.user_id, e.artist_id, COUNT(*) AS escuchas
FROM events_clean e
JOIN usuarios_ok  u ON e.user_id   = u.user_id
JOIN top_artistas t ON e.artist_id = t.artist_id
GROUP BY e.user_id, e.artist_id
"""

con.execute(f"COPY ({consulta_ua}) TO '../data/vistas/vista_usuario_artista.parquet' (FORMAT parquet)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

**Control de la vista.** La *densidad* es el porcentaje de casilleros de la matriz que tienen valor:
pares existentes / (usuarios × artistas). En los sistemas de recomendación suele ser muy baja, por eso
se guardan como matrices dispersas.

In [10]:
resumen_ua = q("""
    SELECT COUNT(*)                  AS pares,
           COUNT(DISTINCT user_id)   AS usuarios,
           COUNT(DISTINCT artist_id) AS artistas,
           MEDIAN(escuchas)          AS mediana_escuchas,
           MAX(escuchas)             AS max_escuchas
    FROM '../data/vistas/vista_usuario_artista.parquet'
""")

resumen_ua["densidad_%"] = resumen_ua["pares"] / (resumen_ua["usuarios"] * resumen_ua["artistas"]) * 100
resumen_ua

,pares,usuarios,artistas,mediana_escuchas,max_escuchas,densidad_%
0,12691416,53271,10000,4.0,181440,2.382425


**B.2 Observaciones:**
- La tabla tiene 12.691.422 pares de 53.271 usuarios y 10.000 artistas. Hay 18 usuarios de la vista 1
  que no escuchan a ninguno de los 10.000 artistas, por eso no aparecen.
- La densidad es 2,4%: de cada 100 casilleros de la matriz, solo 2 tienen algún valor. Sería mucho
  más baja con todos los artistas.
- La mediana es 4 escuchas por par, pero el máximo es 181.440. En el recomendador conviene usar el
  logaritmo de las escuchas para que un artista escuchado miles de veces no tape a los demás.

### B.3 Vista para reglas de asociación: `vista_sesiones`

**Cada fila es un artista dentro de una sesión de escucha.** Cada sesión funciona como una "canasta"
(como en el ejemplo del supermercado de las reglas de asociación) y los artistas son los productos.

**Cómo armamos las sesiones:** si entre dos escuchas seguidas de un usuario pasan más de 30 minutos
(1.800 segundos), empieza una sesión nueva (ver 4.2.e del notebook 01). Los pasos son:
1. Ordenar las escuchas de cada usuario por fecha y calcular cuánto pasó desde la anterior con `LAG`
   (que trae el valor de la fila anterior).
2. Marcar con 1 las escuchas que arrancan una sesión (la primera del usuario o las que vienen después
   de más de 30 minutos).
3. Sumar esas marcas en orden: la suma acumulada da el número de sesión de cada escucha.
4. Quedarnos con los artistas distintos de cada sesión, solo del top 1.000, y solo las sesiones con
   2 artistas o más (con un solo artista no se puede armar ninguna regla).

**Decisiones:**
- **Top 1.000 artistas**, menos que en el recomendador: con artistas poco escuchados el soporte da
  casi cero y no aparecen reglas.
- **El 10% de los usuarios** (`user_id % 10 = 0`, los ids que terminan en 0). Ordenar 350 millones de
  escuchas es lo más pesado de todo el trabajo, y Apriori no aguanta decenas de millones de canastas.
  Con el 10% igual salen muchas sesiones.

In [11]:
consulta_sesiones = """
WITH top_artistas AS (
    SELECT artist_id
    FROM oyentes_artista
    ORDER BY oyentes DESC
    LIMIT 1000
),

-- 1) Segundos desde la escucha anterior del mismo usuario
ordenado AS (
    SELECT user_id, artist_id, "timestamp",
           "timestamp" - LAG("timestamp") OVER (PARTITION BY user_id ORDER BY "timestamp") AS gap
    FROM events_clean
    WHERE user_id IN (SELECT user_id FROM '../data/vistas/vista_usuarios.parquet')
      AND user_id % 10 = 0
),

-- 2) 1 si la escucha empieza una sesión nueva, 0 si sigue la anterior
marcado AS (
    SELECT *,
           CASE WHEN gap IS NULL OR gap > 1800 THEN 1 ELSE 0 END AS nueva
    FROM ordenado
),

-- 3) Suma acumulada de las marcas = número de sesión dentro de cada usuario
con_sesion AS (
    SELECT user_id, artist_id,
           SUM(nueva) OVER (PARTITION BY user_id ORDER BY "timestamp"
                            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS nro_sesion
    FROM marcado
),

-- 4) Artistas distintos por sesión, solo del top, y sesiones con al menos 2 artistas
canasta AS (
    SELECT DISTINCT s.user_id, s.nro_sesion, s.artist_id
    FROM con_sesion s
    JOIN top_artistas t ON s.artist_id = t.artist_id
),
validas AS (
    SELECT user_id, nro_sesion
    FROM canasta
    GROUP BY user_id, nro_sesion
    HAVING COUNT(*) >= 2
)
SELECT c.user_id, c.nro_sesion, c.artist_id
FROM canasta c
JOIN validas v ON c.user_id = v.user_id AND c.nro_sesion = v.nro_sesion
"""

con.execute(f"COPY ({consulta_sesiones}) TO '../data/vistas/vista_sesiones.parquet' (FORMAT parquet)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

**Control de la vista:** cuántas sesiones hay y cuántos artistas tiene cada una.

In [12]:
por_sesion = q("""
    SELECT user_id, nro_sesion, COUNT(*) AS artistas_en_sesion
    FROM '../data/vistas/vista_sesiones.parquet'
    GROUP BY user_id, nro_sesion
""")

print(f"Usuarios en la muestra: {por_sesion['user_id'].nunique():,}")
print(f"Sesiones (canastas):    {len(por_sesion):,}")
print("\nArtistas distintos por sesión:")
print(por_sesion["artistas_en_sesion"].describe(percentiles=[0.5, 0.9, 0.99]).round(1))

Usuarios en la muestra: 5,215
Sesiones (canastas):    1,008,260

Artistas distintos por sesión:
count    1008260.0
mean           3.8
std            3.2
min            2.0
50%            3.0
90%            7.0
99%           17.0
max          112.0
Name: artistas_en_sesion, dtype: float64


**B.3 Observaciones:**
- Con los 5.215 usuarios de la muestra salen 1.008.260 sesiones, más que suficientes para buscar reglas.
- Una sesión típica tiene 3 artistas distintos (la mediana). El 90% tiene 7 o menos y el 99%, 17 o menos.
- Hay algunas sesiones muy largas, con hasta 112 artistas. Seguramente son usuarios que dejaron la música
  sonando muchas horas seguidas. Son pocas, así que las dejamos.

## Resumen

| Vista | Cada fila es... | Filtros | Tamaño |
|---|---|---|---|
| `vista_usuarios` | un usuario | 100 escuchas con audio o más | 53.289 usuarios |
| `vista_usuario_artista` | un usuario con un artista | usuarios de la vista 1, top 10.000 artistas | 12.691.422 pares |
| `vista_sesiones` | un artista en una sesión | 10% de los usuarios, top 1.000 artistas, sesiones con 2 artistas o más | 1.008.260 sesiones |

Las tres vistas quedan en `data/vistas/` y son las que usan los notebooks de clustering, recomendación
y reglas de asociación.